1.
<p>Create a notebook named 'fest_collect.ipynb' and a SQLite database 'fest.db' with `sqlite3.connect`. Write a 400-row `registrations` table (reg_id, student_id S001 to S400, event from Dance, Coding, Quiz, Gaming, Music, college from 6 invented names, ticket_inr 0 to 500 in steps of 50, paid Yes/No) using seed 42 and `to_sql(index=False)`. Read the paid rows back with `pd.read_sql_query` and again with `pd.read_sql`, printing both shapes. Constraint: Use only a simple `SELECT ... WHERE` query.</p>
2.
<p>Write a nested, API-style response of 15 fictional sponsors to 'sponsors_api.json' with `json.dump`, shaped as `{'status': 'ok', 'data': [{'sponsor': ..., 'amount_inr': ..., 'contact': {'city': ..., 'tier': ...}}]}`. Load it with `json.load` and flatten `data` into a `sponsors` DataFrame with `pd.json_normalize`; print its columns, which must include `contact.city` and `contact.tier`.</p>
3.
<p>Create 'fest_events.csv' with 5 rows (event, venue, day from 1 to 3, budget_inr) for your five events, push it to your public GitHub repo 'ds_analysis_portfolio', and load it into `events` with `pd.read_csv` using its raw.githubusercontent.com web address. Print `events.shape`. Hint: On the file's GitHub page click Raw, then copy the address from the browser bar.</p>
4.
<p>Split `registrations` into three files, 'fest_data/day1.csv', 'day2.csv' and 'day3.csv' (130, 140 and 130 rows), creating the folder with `Path('fest_data').mkdir(exist_ok=True)`. Read them back with `sorted(Path('fest_data').glob('day*.csv'))` and `pd.concat(ignore_index=True)`, then `merge` with `events` on event (`how='left'`, `validate='m:1'`) into 'fest_master.csv'. Assert 400 rows and no missing venue. Run Restart & Run All and push to 's05_fest_collect' in your GitHub repo 'ds_analysis_portfolio'.</p>

In [2]:
import numpy as np
import pandas as pd
import sqlite3
import json
from pathlib import Path

In [3]:
""" Task 1
"""
np.random.seed(42)

# Create 400 registrations
registrations = pd.DataFrame({
    "reg_id": range(1, 401),
    "student_id": [f"S{i:03d}" for i in range(1, 401)],
    "event": np.random.choice(
        ["Dance", "Coding", "Quiz", "Gaming", "Music"], 400
    ),
    "college": np.random.choice(
        ["ABC College", "XYZ College", "Sunrise College",
         "Greenfield College", "National College", "Modern College"],
        400
    ),
    "ticket_inr": np.random.choice(
        range(0, 501, 50), 400
    ),
    "paid": np.random.choice(
        ["Yes", "No"], 400
    )
})

print(registrations.head())
print("Shape:", registrations.shape)

   reg_id student_id   event           college  ticket_inr paid
0       1       S001  Gaming   Sunrise College         500   No
1       2       S002   Music  National College         500   No
2       3       S003    Quiz    Modern College         500  Yes
3       4       S004   Music       ABC College         100  Yes
4       5       S005   Music  National College         400   No
Shape: (400, 6)


In [4]:
# Connect to database
conn = sqlite3.connect("fest.db")

# Save DataFrame into SQLite table
registrations.to_sql(
    "registrations",
    conn,
    if_exists="replace",
    index=False
)

print("Database and registrations table created!")

Database and registrations table created!


In [5]:
query = """
SELECT * FROM registrations
WHERE paid = 'Yes'
"""

# Read using read_sql_query
paid_query = pd.read_sql_query(query, conn)

print("Using read_sql_query:")
print(paid_query.shape)

# Read using read_sql
paid_sql = pd.read_sql(query, conn)

print("Using read_sql:")
print(paid_sql.shape)

# Close database connection
conn.close()

Using read_sql_query:
(210, 6)
Using read_sql:
(210, 6)


In [6]:
# Create sponsor data
sponsor_data = {
    "status": "ok",
    "data": []
}

for i in range(1, 16):
    sponsor = {
        "sponsor": f"Sponsor {i}",
        "amount_inr": i * 5000,
        "contact": {
            "city": ["Ahmedabad", "Surat", "Vadodara",
                     "Rajkot", "Gandhinagar"][i % 5],
            "tier": ["Gold", "Silver", "Platinum"][i % 3]
        }
    }

    sponsor_data["data"].append(sponsor)

# Save JSON file
with open("sponsors_api.json", "w") as file:
    json.dump(sponsor_data, file, indent=4)

print("Sponsors JSON file created!")

Sponsors JSON file created!


In [7]:
# Load JSON file
with open("sponsors_api.json", "r") as file:
    loaded_data = json.load(file)

# Flatten nested data
sponsors = pd.json_normalize(loaded_data["data"])

# Display columns
print("Sponsor DataFrame:")
print(sponsors.head())

print("Columns:")
print(sponsors.columns.tolist())

Sponsor DataFrame:
     sponsor  amount_inr contact.city contact.tier
0  Sponsor 1        5000        Surat       Silver
1  Sponsor 2       10000     Vadodara     Platinum
2  Sponsor 3       15000       Rajkot         Gold
3  Sponsor 4       20000  Gandhinagar       Silver
4  Sponsor 5       25000    Ahmedabad     Platinum
Columns:
['sponsor', 'amount_inr', 'contact.city', 'contact.tier']


In [8]:
""" Task 3
"""
events_data = {
    "event": ["Dance", "Coding", "Quiz", "Gaming", "Music"],
    "venue": [
        "Main Auditorium",
        "Computer Lab",
        "Seminar Hall",
        "Gaming Arena",
        "Open Ground"
    ],
    "day": [1, 1, 2, 2, 3],
    "budget_inr": [15000, 20000, 10000, 18000, 25000]
}

# Create events DataFrame
events = pd.DataFrame(events_data)

# Save CSV file
events.to_csv("fest_events.csv", index=False)

print(events)
print("Festival events CSV created!")

    event            venue  day  budget_inr
0   Dance  Main Auditorium    1       15000
1  Coding     Computer Lab    1       20000
2    Quiz     Seminar Hall    2       10000
3  Gaming     Gaming Arena    2       18000
4   Music      Open Ground    3       25000
Festival events CSV created!


In [10]:
"""Task 4
"""

Path("fest_data").mkdir(exist_ok=True)


day1 = registrations.iloc[:130]
day2 = registrations.iloc[130:270]
day3 = registrations.iloc[270:400]


day1.to_csv("fest_data/day1.csv", index=False)
day2.to_csv("fest_data/day2.csv", index=False)
day3.to_csv("fest_data/day3.csv", index=False)

print("Three festival files created!")

print("Day 1:", day1.shape)
print("Day 2:", day2.shape)
print("Day 3:", day3.shape)

# Find all day CSV files in sorted order
files = sorted(Path("fest_data").glob("day*.csv"))

# Read and combine files
data_list = []

for file in files:
    data = pd.read_csv(file)
    data_list.append(data)

combined = pd.concat(data_list, ignore_index=True)

print("Combined Data:")
print(combined.head())

print("Combined Shape:", combined.shape)


# Merge using event column
fest_master = pd.merge(
    combined,
    events,
    on="event",
    how="left",
    validate="m:1"
)

# Save final master file
fest_master.to_csv("fest_master.csv", index=False)

print(fest_master.head())
print("Master Shape:", fest_master.shape)

# Check that there are 400 rows
assert len(fest_master) == 400

# Check that no venue is missing
assert fest_master["venue"].isnull().sum() == 0

print("All checks passed!")
print("Total Rows:", len(fest_master))
print("Missing Venues:", fest_master["venue"].isnull().sum())

Three festival files created!
Day 1: (130, 6)
Day 2: (140, 6)
Day 3: (130, 6)
Combined Data:
   reg_id student_id   event           college  ticket_inr paid
0       1       S001  Gaming   Sunrise College         500   No
1       2       S002   Music  National College         500   No
2       3       S003    Quiz    Modern College         500  Yes
3       4       S004   Music       ABC College         100  Yes
4       5       S005   Music  National College         400   No
Combined Shape: (400, 6)
   reg_id student_id   event           college  ticket_inr paid         venue  \
0       1       S001  Gaming   Sunrise College         500   No  Gaming Arena   
1       2       S002   Music  National College         500   No   Open Ground   
2       3       S003    Quiz    Modern College         500  Yes  Seminar Hall   
3       4       S004   Music       ABC College         100  Yes   Open Ground   
4       5       S005   Music  National College         400   No   Open Ground   

   day  bud